In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import fetch_california_housing
from joblib import dump
import math
from scipy.io import loadmat

import matplotlib.pyplot as plt


In [ ]:
#read a matlab struct from a mat file and save it into a pd dataframe

mat = loadmat('./microstructures_object_array_final.mat')
raw_rows = mat['final_microstructure_array'][0]

column_names = raw_rows[0].dtype.names

data = []

for raw_row in raw_rows:
    row=[]
    for array in raw_row:
        if len(array[0]) == 1:
            row.append(array[0][0])
        else:
            row.append(array[0])

    data.append(row)

data

df =  pd.DataFrame(data, columns= column_names )


num_rows = df.shape[0]

# Separate the last 5 rows into a validation set
validation_df = df.iloc[num_rows - 6:]

# The rest of the rows go into the training set
training_df = df.iloc[:num_rows - 6]

df = training_df

df["R11"] = df["yield_stress_XX"]/df["yield_stress_ZZ"]

training_df

In [ ]:
plt.scatter(df['lath_thickness'], df['yield_stress_ZZ'], color = 'lightcoral')
plt.title('lath_thickness vs yield_stress_ZZ')
plt.xlabel('lath_thickness')
plt.ylabel('yield_ZZ')
plt.box(False)
plt.show()



In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot the data
sc = ax.scatter(df['f_alpha'], df['lath_thickness'], df['Ez'], c=df['f_beta'])

# Set labels and title
ax.set_xlabel('f_alpha')
ax.set_ylabel('alpha_lath_thickness')
ax.set_zlabel('yield_stress_ZZ')
ax.set_title('3D Scatter Plot')

# Add a colorbar
cbar = fig.colorbar(sc, label='f_beta')
cbar.set_ticks([0.0, 0.1, 0.2])  # Set colorbar ticks to 0.0, 0.1, and 0.2


# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/plot.jpeg')



In [ ]:
import seaborn as sns

# Create a PairGrid
g = sns.PairGrid(df, y_vars=['R11'], x_vars=['lath_thickness', 'f_alpha', 'f_beta', 'f_mart'], height=4)

# Map a scatter plot to the upper triangle, color points by 'f_beta'
g.map(sns.scatterplot, hue=df['f_mart'], palette='coolwarm')

# Add a legend
plt.legend(title='f_beta', loc='upper left')

# Show the plot
plt.show()

In [ ]:
# Polynomial multi-variate regression

# Select appropriate features from the dataset
X = df[['lath_thickness','f_alpha','f_beta']].values

properties = ['yield_stress_ZZ','yield_stress_XX','yield_stress_XY','yield_stress_XZ','Ex','Ez','Gxy','Gxz','Pxz','Pyx']

property = 'yield_stress_XX'
y = df[property].values

# Polynomial features
poly = PolynomialFeatures(degree=5)
X_poly = poly.fit_transform(X)

# Linear Regression model
model = LinearRegression()
model.fit(X_poly, y)

r_sq = model.score(X_poly,y)
print(f"coefficient of determination: {r_sq}")

print(f"intercept: {model.intercept_}")
print(f"slope: {model.coef_}")


# Calculate error with validation data set:

error = []

for index, row in validation_df.iterrows():
    
    print(row['f_alpha'] + row['f_beta'])
    
    new_data = np.array([row['lath_thickness'], row['f_alpha'], row['f_beta']]).reshape(1,-1)  # Example values
    new_data_poly = poly.transform(new_data)


    # Predicting the price
    predicted_y = model.predict(new_data_poly)

    error.append( (predicted_y[0] - row[property]) / row[property] * 100)

print('error_percentage',error)


In [ ]:
## Create Regression model for all independent variables and save them:

from joblib import load

# Select appropriate features from the dataset
X = df[['lath_thickness','f_alpha','f_beta']].values


properties = ['yield_stress_ZZ','yield_stress_XX','yield_stress_XY','yield_stress_XZ','Ex','Ez','Gxy','Gxz','Pxz','Pyx']

for property in properties:
    y = df[property].values

    # Polynomial features
    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X)

    # Linear Regression model
    model = LinearRegression()
    model.fit(X_poly, y)

    dump(model, f'../data/models_saved/{property}_model.joblib')
    dump(poly, f'../data/models_saved/{property}_poly_transformer.joblib')


In [ ]:
hardening = df['yield_stress_hardening_ZZ'].values

for i in range(0,len(hardening[0])):
    # values = [sub_arr[0] for sub_arr in hardening]
    stress_values_this_plastic = [sub_arr[i] for sub_arr in hardening]
    poly = PolynomialFeatures(degree=2)
    X_poly = poly.fit_transform(X)

    # Linear Regression model
    model = LinearRegression()
    model.fit(X_poly, stress_values_this_plastic)

    dump(model, f'../data/models_saved/hardening_{i}_model.joblib')
    dump(poly, f'../data/models_saved/hardening_{i}_poly_transformer.joblib')



In [ ]:
# Read Plastic Strain data and calculate average plastic strain
plastic_strain = df['plastic_strain_ZZ'].values

plastic_strain_average_list = []

for i in range(len(plastic_strain[0])):

    plastic_values_this_strain = [sub_array[i] for sub_array in plastic_strain]
    plastic_strain_average_list.append( sum(plastic_values_this_strain) / len(plastic_values_this_strain) )

# Save plastic_strain_average_list

np.save('../data/plastic_strain_average_list.npy', plastic_strain_average_list)

plastic_strain_average_list

In [ ]:
## Load Model and predict properties for a given microstructure

properties = ['yield_stress_ZZ','yield_stress_XX','yield_stress_XY','yield_stress_XZ','Ex','Ez','Gxy','Gxz','Pxz','Pyx']

pred_prop_dict = {}

for property in properties:

    # Load the model and the transformer
    model = load(f'../data/models_saved/{property}_model.joblib')
    poly = load(f'../data/models_saved/{property}_poly_transformer.joblib')

    # New data
    new_data = np.array([[0.4,0.33,0.09]])  # Example values

    # Transform the new data using the loaded transformer
    new_data_poly = poly.transform(new_data)

    # Use the loaded model to make a prediction
    predicted_y = model.predict(new_data_poly)

    pred_prop_dict[property] = predicted_y[0]


pred_prop_dict['sigma_0'] = []
for i in range(0,len(plastic_strain_average_list)):

    # Load the model and the transformer
    model = load(f'../data/models_saved/hardening_{i}_model.joblib')
    poly = load(f'../data/models_saved/{property}_poly_transformer.joblib')

    # New data
    new_data = np.array([[0.4,0.33,0.09]])  # Example values
    # Transform the new data using the loaded transformer
    new_data_poly = poly.transform(new_data)

    # Use the loaded model to make a prediction
    predicted_y = model.predict(new_data_poly)
    pred_prop_dict['sigma_0'].append(predicted_y[0])

pred_prop_dict['strain_0'] = plastic_strain_average_list

pred_prop_dict

In [ ]:
#write material INP:

import importlib
import write_material_inp

importlib.reload(write_material_inp)

write_material_inp.write_inp(pred_prop_dict, '../data/Val_2_interp.inp')

# ## Write Material info in Abaqus inp format:

# #Calculate Hill Parameters from yield stresses
# sigma_0 = pred_prop_dict['yield_stress_ZZ']
# R11 = pred_prop_dict['yield_stress_XX']/sigma_0
# R22 = R11
# R33 = 1
# R12 = pred_prop_dict['yield_stress_XY']/(sigma_0/math.sqrt(3))
# R13 = pred_prop_dict['yield_stress_XZ']/(sigma_0/math.sqrt(3))
# R23 = R13

# hardening_string = ''
# for i in range(len(pred_prop_dict['strain_0'])):
#     hardening_string += f"\n{pred_prop_dict['sigma_0'][i]},{pred_prop_dict['strain_0'][i]}"

# #Write Material info in Abaqus inp format:
# Material_Abaqus_text =f'''**
# *Material, name=Material-1
# *Elastic, type=ENGINEERING CONSTANTS
# {pred_prop_dict['Ex']},{pred_prop_dict['Ex']},{pred_prop_dict['Ez']}, {pred_prop_dict['Pyx']}, {pred_prop_dict['Pxz']}, {pred_prop_dict['Pxz']}, {pred_prop_dict['Gxy']}, {pred_prop_dict['Gxz']}
# {pred_prop_dict['Gxz']},
# *Plastic{hardening_string}
# *Potential
# {R11}, {R22},  {R33},  {R12},  {R12},  {R23}'''


# with open('../data/material.inp', 'w') as f:
#     # Write the string to the file
#     f.write(Material_Abaqus_text)


In [ ]:
## Write INP for a CP result
import importlib
import write_material_inp

importlib.reload(write_material_inp)

# Get the row number
# row_number = 0 #CP_HT900
row_number = -1

# Convert the row into a dictionary
row_dict = validation_df.iloc[row_number].to_dict()

row_dict["strain_0"] = row_dict['plastic_strain_ZZ']
row_dict["sigma_0"] = row_dict['yield_stress_hardening_ZZ']


write_material_inp.write_inp(row_dict, '../data/material_second_validation_CP.inp')


In [ ]:
row_dict

In [ ]:
## Linear_Interpolation in 3D unstructured grid

import numpy as np
from scipy.interpolate import griddata

property = 'yield_stress_ZZ'

# Define the points where you know the function values
points = df[['lath_thickness','f_alpha','f_beta']].values # 100 points in 3D
values = df[property].values

# Define the point where you want to interpolate
interp_point = np.array([0.5, 0.0, 0.0])

# Perform the interpolation
interp_value = griddata(points, values, [interp_point], method='linear')

interp_value

# Calculate error with validation data set:

error = []

for index, row in validation_df.iterrows():

    
    interp_point = np.array([row['lath_thickness'], row['f_alpha'], row['f_beta']]).reshape(1,-1)  # Example values
    print(interp_point)
    interp_value = griddata(points, values, [interp_point], method='linear')


    predicted_y = interp_value[0]
    error.append( ( (predicted_y - row[property]) / row[property] ) * 100 )

print('error_percentage',error)


In [ ]:
#RBF Interpolation

import numpy as np
from scipy.interpolate import RBFInterpolator


property = 'yield_stress_ZZ'

# Define the points where you know the function values
points = df[['lath_thickness','f_alpha','f_beta']].values # 100 points in 3D
values = df[property].values

# Define the point where you want to interpolate
rbf = RBFInterpolator(points, values, kernel = 'quintic')


# interp_value = rbf([[2.35, 0.8, 0.1]])

# Calculate error with validation data set:
error = []

for index, row in validation_df.iterrows():

    
    interp_point = np.array([row['lath_thickness'], row['f_alpha'], row['f_beta']]).reshape(1,-1) 
    interp_value = rbf( interp_point )

    predicted_y = interp_value[0]

    print(predicted_y, row[property])

    error.append( ( (predicted_y - row[property]) / row[property] ) * 100 )

print('error_percentage',error)




In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 16})


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    ax.scatter(df['f_alpha'][mask], df['lath_thickness'][mask], df['yield_stress_ZZ'][mask], color=colors[i], label=labels[i])

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y BD}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y BD}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(title='$f_\\beta$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_BD.jpeg', dpi=600)



In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 16})


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    ax.scatter(df['f_alpha'][mask], df['lath_thickness'][mask], df['yield_stress_XX'][mask], color=colors[i], label=labels[i])

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y XX}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y XX}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(title='$f_\\beta$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_XX.jpeg', dpi=600)



In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 16})


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red', 'black', 'purple', 'brown']  # Modify this list to match your data
labels = ['0', '0.2', '0.4', '0.6', '0.8', '1.0']  # Modify this list to match your data

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_alpha'].unique()):
    mask = df['f_alpha'] == value
    ax.scatter(df['f_alpha'][mask], df['lath_thickness'][mask], df['yield_stress_ZZ'][mask], color=colors[i], label=labels[i])

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y BD}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y BD}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(title='$f_\\alpha$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_BD_alpha.jpeg', dpi=600)




In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 16})


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red', 'black', 'purple', 'brown']  # Modify this list to match your data
labels = ['0', '0.2', '0.4', '0.6', '0.8', '1.0']  # Modify this list to match your data

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_alpha'].unique()):
    mask = df['f_alpha'] == value
    ax.scatter(df['f_alpha'][mask], df['lath_thickness'][mask], df['yield_stress_XX'][mask], color=colors[i], label=labels[i])

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y XX}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y XX}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(title='$f_\\alpha$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_XX_alpha.jpeg', dpi=600)

In [ ]:


import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({'font.size': 16})


# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    ax.scatter(df['f_alpha'][mask], df['lath_thickness'][mask], df['Ex'][mask], color=colors[i], label=labels[i])

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$Exx(GPa)$', fontsize=20, labelpad=10)
ax.set_title('$Exx}$', fontsize=22)

# ax.set_zlim([650, 1050])

# Add a legend with a title
legend = ax.legend(title='$f_\\beta$')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/Exx_BD.jpeg', dpi=600)




In [ ]:
import matplotlib.patches as mpatches

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Create a list to store the patches for the legend
patches = []

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    x = df['f_alpha'][mask]
    y = df['lath_thickness'][mask]
    z = df['yield_stress_ZZ'][mask]

    # Create a grid of x and y values
    xi = np.linspace(x.min(), x.max(), 100)
    yi = np.linspace(y.min(), y.max(), 100)
    xi, yi = np.meshgrid(xi, yi)

    # Interpolate z values for the grid
    zi = griddata((x, y), z, (xi, yi), method='cubic')

    # Plot the surface
    ax.plot_surface(xi, yi, zi, color=colors[i], alpha=0.4)

    # Create a patch for the legend
    patches.append(mpatches.Patch(color=colors[i], label=labels[i]))

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y BD}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y BD}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(handles=patches, title='$f_\\beta$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_BD._surf.jpeg', dpi=600)



In [ ]:
import matplotlib.patches as mpatches

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Create a list to store the patches for the legend
patches = []

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    x = df['f_alpha'][mask]
    y = df['lath_thickness'][mask]
    z = df['yield_stress_XX'][mask]

    # Create a grid of x and y values
    xi = np.linspace(x.min(), x.max(), 100)
    yi = np.linspace(y.min(), y.max(), 100)
    xi, yi = np.meshgrid(xi, yi)

    # Interpolate z values for the grid
    zi = griddata((x, y), z, (xi, yi), method='cubic')

    # Plot the surface
    ax.plot_surface(xi, yi, zi, color=colors[i], alpha=0.4)

    # Create a patch for the legend
    patches.append(mpatches.Patch(color=colors[i], label=labels[i]))

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y XX}(MPa)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y XX}$', fontsize=22)

ax.set_zlim([650, 900])

# Add a legend with a title
legend = ax.legend(handles=patches, title='$f_\\beta$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_XX_surf.jpeg', dpi=600)



In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each point based on yield_stress_XX
colors = plt.cm.viridis((df['yield_stress_XX'] - df['yield_stress_XX'].min()) / (df['yield_stress_XX'].max() - df['yield_stress_XX'].min()))

# Plot the data
sc = ax.scatter(df['f_alpha'], df['f_beta'], df['lath_thickness'], c=colors, alpha=1)

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$f_{\\beta}$', fontsize=20, labelpad=10)
ax.set_zlabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y XX}$', fontsize=22)

# Add a colorbar
fig.colorbar(sc, label='$\\sigma_{y XX}(MPa)$')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/sigma_XX_contour.jpeg', dpi=600)



In [ ]:
import matplotlib.patches as mpatches

# Create a 3D scatter plot
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

# Define the colors for each value of f_beta
colors = ['blue', 'green', 'red']  # Modify this list to match your data
labels = ['0', '0.1', '0.2']  # Modify this list to match your data

# Create a list to store the patches for the legend
patches = []

# Plot the data for each value of f_beta
for i, value in enumerate(df['f_beta'].unique()):
    mask = df['f_beta'] == value
    x = df['f_alpha'][mask]
    y = df['lath_thickness'][mask]
    z = df['R11'][mask]

    # Create a grid of x and y values
    xi = np.linspace(x.min(), x.max(), 100)
    yi = np.linspace(y.min(), y.max(), 100)
    xi, yi = np.meshgrid(xi, yi)

    # Interpolate z values for the grid
    zi = griddata((x, y), z, (xi, yi), method='cubic')

    # Plot the surface
    ax.plot_surface(xi, yi, zi, color=colors[i], alpha=0.4)

    # Create a patch for the legend
    patches.append(mpatches.Patch(color=colors[i], label=labels[i]))

# Set labels and title
ax.set_xlabel('$f_{\\alpha }$', fontsize=20, labelpad=10)
ax.set_ylabel('$t_{\\alpha}(\\mu m)$', fontsize=20, labelpad=10)
ax.set_zlabel('$\\sigma_{y XX} / \\sigma_{y BD}$', fontsize=20, labelpad=10)
ax.set_title('$\\sigma_{y XX} / \\sigma_{y BD}$', fontsize=22)

# ax.set_zlim([650, 1050])

# Add a legend with a title
legend = ax.legend(handles=patches, title='$f_\\beta$', loc='upper left')

# Change the font size of the legend title
legend.get_title().set_fontsize('20')

# Show the plot
plt.show()

# Save the plot to a file
fig.savefig('../data/anisotropy_surf.jpeg', dpi=600)